In [2]:
from PIChartFunctions import PiDataRetrival, calculate_5_year_average # The 5 year data has a cut-off '2023-07-01' ?????
import pandas as pd
import PIconnect as PI
from datetime import datetime, timedelta
import pytz
import plotly.graph_objects as go
from plotformating import line_chart_formatting
# 1) fetch raw PI data

# if pi_name.strip().lower() == 'mount burr wwwtp':
#     pi_name = 'Mount Burr WWTP'

raw = PiDataRetrival("Finger Point WWTP")



# 3) prepare 5‐year pivot
raw['Timestamp'] = pd.to_datetime(raw['Timestamp'])
end_dt = pd.to_datetime('01/07/2025', utc=True)
start_time = end_dt - timedelta(days=1825)
mask = (raw['Timestamp'] >= start_time) & (raw['Timestamp'] <= end_dt)
data_5_year  = raw.loc[mask].copy()
data_5_year['MonthNum']   = data_5_year['Timestamp'].dt.month
data_5_year['MonthLabel'] = data_5_year['Timestamp'].dt.strftime('%b')

data_5_year['Financial Year'] = data_5_year['Timestamp'].apply(
    lambda x: f"{x.year}-{x.year+1}" if x.month >= 7 else f"{x.year-1}-{x.year}"
)

rename_map = {
    '1 - Influent':                 'Influent',
    '2 - Effluent':                 'Effluent',
    '3 - Reuse':                    'Reuse',
    '5 - Discharge to Environment': 'Discharge'
}

df_data_5_year = (
    data_5_year
    .pivot_table(
    index='Financial Year',
    columns='Attribute',
    values='Value',
    aggfunc='sum'
    )
    .rename(columns=rename_map)
    .reset_index()
)


TypeError: Invalid comparison between dtype=datetime64[ns] and Timestamp

In [8]:
from PIChartFunctions import PiDataRetrival
import pandas as pd

# 1) fetch raw PI data
site_name = "Finger Point WWTP"
raw = PiDataRetrival(site_name).copy()
raw["Timestamp"] = pd.to_datetime(raw["Timestamp"])

# 2) Replacement data for Attribute = 3 - Reuse

TARGET_SITE = "Finger Point WWTP"
# Only build/apply reuse adjustments when we are on the target site
if site_name == TARGET_SITE:
    reuse_data = pd.DataFrame({
        "Site": [TARGET_SITE] * 4,
        "Attribute": ["3 - Reuse"] * 4,
        "Timestamp": ["2024-07-01", "2024-12-01", "2025-04-01", "2025-06-01"],
        "Value": [0.14, 0.42, 0.01, 0.03]
    })
    reuse_data["Timestamp"] = pd.to_datetime(reuse_data["Timestamp"])

    # Step 1: replace "3 - Reuse" for this site at those timestamps
    ts_match = set(reuse_data["Timestamp"])
    mask_old_reuse = (
        (raw["Site"] == TARGET_SITE) &
        (raw["Attribute"] == "3 - Reuse") &
        (raw["Timestamp"].isin(ts_match))
    )
    raw = raw[~mask_old_reuse]
    raw = pd.concat([raw, reuse_data], ignore_index=True)

    # Step 2: subtract reuse from "5 - Discharge to Environment" (match on Site+Timestamp)
    mask_discharge = (
        (raw["Site"] == TARGET_SITE) &
        (raw["Attribute"] == "5 - Discharge to Environment")
    )
    sub = (
        raw.loc[mask_discharge, ["Site", "Timestamp", "Value"]]
           .merge(reuse_data[["Site", "Timestamp", "Value"]]
                  .rename(columns={"Value": "ReuseValue"}),
                  on=["Site", "Timestamp"], how="left")
    )
    raw.loc[mask_discharge, "Value"] = (sub["Value"] - sub["ReuseValue"].fillna(0)).values
else:
    # Not the Finger Point dataset: skip any reuse adjustments entirely
    pass

print(raw[raw["Timestamp"].isin(ts_match)]) # check replacement worked


                  Site                     Attribute  Timestamp       Value
46   Finger Point WWTP                  1 - Influent 2024-07-01  162.568344
51   Finger Point WWTP                  1 - Influent 2024-12-01  163.525024
55   Finger Point WWTP                  1 - Influent 2025-04-01  163.552536
57   Finger Point WWTP                  1 - Influent 2025-06-01  167.606628
105  Finger Point WWTP                  2 - Effluent 2024-07-01  192.905273
110  Finger Point WWTP                  2 - Effluent 2024-12-01  185.115738
114  Finger Point WWTP                  2 - Effluent 2025-04-01  182.031525
116  Finger Point WWTP                  2 - Effluent 2025-06-01  195.858627
164  Finger Point WWTP  5 - Discharge to Environment 2024-07-01  192.765273
169  Finger Point WWTP  5 - Discharge to Environment 2024-12-01  184.695738
173  Finger Point WWTP  5 - Discharge to Environment 2025-04-01  182.021525
175  Finger Point WWTP  5 - Discharge to Environment 2025-06-01  195.828627
177  Finger 

In [2]:
from report_functions import (
    load_sites,
    load_report_details,
    write_site_information,
    create_flow_charts,
    get_testing_components_data,
    get_site_incidents_data,
    get_comments_data,
    load_monitoring_plan_requirements,
    write_monitoring_program_sheet
)
from datetime import datetime
import pandas as pd

start_dt = pd.to_datetime('01-07-2024', dayfirst=True)
end_dt   = pd.to_datetime('30-06-2025', dayfirst=True)

inc_df = get_site_incidents_data("Murray Bridge WWTP", start_dt, end_dt)
inc_df

,Date,Incident ID,Location Description,Incident Detailed Description,Primary Incident Classification,Incident Status,Incident Date Time
774,2024-2025,INC-0063755,Heathfield WWTP,Heathfield Activated Sludge WWTP (Brick Kiln R...,Type 3,Closed,2024-07-01 01:00:00
775,2024-2025,INC-0063763,Heathfield WWTP,"Blower 2 VSD fault, VSD showing ""input phase l...",Type 3,Closed,2024-07-02 10:00:00
777,2024-2025,INC-0063774,Hahndorf Activated Sludge WWTP (Ambleside Rd),Hahndorf Activated Sludge WWTP (Ambleside Rd):...,Type 3,Closed,2024-07-02 04:59:00
778,2024-2025,INC-0063781,Pallamana Wastewater Treatment Plant (Wilkin Rd),Pallamana Wastewater Treatment Plant (Wilkin R...,Type 3,Closed,2024-07-04 10:41:00
779,2024-2025,INC-0063897,Hahndorf Activated Sludge WWTP (Ambleside Rd),Hahndorf Activated Sludge WWTP (Ambleside Rd):...,Type 3,Closed,2024-07-17 10:40:00
...,...,...,...,...,...,...,...
942,2024-2025,INC-0067535,Finger Point WWTP,Finger Point Activated Sludge WWTP (Hutt Bay R...,Type 3,Open,2025-06-25 08:00:00
943,2024-2025,INC-0067541,Naracoorte WWTP (Treatment Works Rd),Sludge room PLC is redundant with no spare par...,Type 3,Open,2025-06-24 14:00:00
944,2024-2025,INC-0067546,Finger Point WWTP,Capital project to replace Finger Point WWTP P...,Type 3,Open,2025-06-26 14:27:00
945,2024-2025,INC-0067608,Port Augusta Prison WWTP,Aerator is out of service at Port Augusta Pris...,Type 3,Open,2025-06-17 12:00:00


In [4]:
df_tc, tc_images,df_tc_5_year,tc_images_5y = get_testing_components_data("Murray Bridge WWTP", start_dt, end_dt)
df_tc.head()

KeyError: 'Timestamp'

In [5]:
import pandas as pd
import pytz

# --- 0) Clean columns and coerce to datetime TZ-aware ---
raw = PiDataRetrival("Finger Point WWTP").copy()
raw.rename(columns=lambda c: c.strip(), inplace=True)  # remove trailing spaces in headers

adtz = pytz.timezone("Australia/Adelaide")

# Make tz-aware UTC; invalid parse -> NaT
raw['Timestamp'] = pd.to_datetime(raw['Timestamp'], errors='coerce', utc=True)

# Optional: inspect bad rows
bad = raw['Timestamp'].isna().sum()
if bad:
    print(f"[Info] {bad} Timestamp values could not be parsed and became NaT.")

# Convert to local time for FY boundaries
raw['Timestamp'] = raw['Timestamp'].dt.tz_convert(adtz)

# --- 1) Define as-of date (local) and compute FY labels (A-JUN) ---
end_dt = pd.Timestamp('2025-07-01', tz=adtz)

raw['FY_period'] = raw['Timestamp'].dt.to_period('A-JUN')        # e.g. 2025 means FY 2024-07-01..2025-06-30
raw['Financial Year'] = raw['FY_period'].apply(lambda p: f"{p.year-1}-{p.year}")

# Last 5 FYs, anchored to end_dt-1day to avoid spilling into next FY on July 1
asof_fy_end_year = (end_dt - pd.Timedelta(days=1)).to_period('A-JUN').year
last5 = [f"{y-1}-{y}" for y in range(asof_fy_end_year-4, asof_fy_end_year+1)]

data_5_year = raw[raw['Financial Year'].isin(last5)].copy()
data_5_year['MonthNum']   = data_5_year['Timestamp'].dt.month
data_5_year['MonthLabel'] = data_5_year['Timestamp'].dt.strftime('%b')

# --- 2) Pivot and rename ---
rename_map = {
    '1 - Influent':                 'Influent',
    '2 - Effluent':                 'Effluent',
    '3 - Reuse':                    'Reuse',
    '5 - Discharge to Environment': 'Discharge'
}

df_data_5_year = (
    data_5_year
      .pivot_table(index='Financial Year', columns='Attribute', values='Value', aggfunc='sum')
      .rename(columns=rename_map)
      .reset_index()
)

# Order FY rows chronologically
fy_cat = pd.Categorical(df_data_5_year['Financial Year'], categories=last5, ordered=True)
df_data_5_year = df_data_5_year.assign(**{'Financial Year': fy_cat}).sort_values('Financial Year').reset_index(drop=True)
df_data_5_year

C:\Users\ka005254\AppData\Local\Temp\ipykernel_34880\1342816704.py:24: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  raw['FY_period'] = raw['Timestamp'].dt.to_period('A-JUN')        # e.g. 2025 means FY 2024-07-01..2025-06-30
C:\Users\ka005254\AppData\Local\Temp\ipykernel_34880\1342816704.py:24: FutureWarning: 'A-JUN' is deprecated and will be removed in a future version, please use 'Y-JUN' instead.
  raw['FY_period'] = raw['Timestamp'].dt.to_period('A-JUN')        # e.g. 2025 means FY 2024-07-01..2025-06-30
C:\Users\ka005254\AppData\Local\Temp\ipykernel_34880\1342816704.py:28: UserWarning: Converting to Period representation will drop timezone information.
  asof_fy_end_year = (end_dt - pd.Timedelta(days=1)).to_period('A-JUN').year
C:\Users\ka005254\AppData\Local\Temp\ipykernel_34880\1342816704.py:28: FutureWarning: 'A-JUN' is deprecated and will be removed in a future version, please use 'Y-JUN' instead.
  asof_fy_end_year = (end_dt - p

Attribute,Financial Year,Influent,Effluent,Discharge
0,2020-2021,1613.780014,1630.819977,1630.819977
1,2021-2022,1908.578369,2186.049683,2186.049683
2,2022-2023,1987.306839,2210.974976,2210.974976
3,2023-2024,1986.876404,2214.181946,2214.181946
4,2024-2025,1952.689880,2220.641098,2220.641098


In [ ]:
import calendar

import os
import csv
import pandas as pd
from datetime import datetime,timedelta
import io
import pandas as pd
import plotly.graph_objects as go
from waterscopequery import waterscope_data
import pyodbc
import pytz


raw_5yr  = waterscope_data(conn, sampling_data, report_name, fy_start - 5, 5)

In [1]:
from PIChartFunctions import PiDataRetrival
import pandas as pd

# 1) fetch raw PI data
site_name = "Murray Bridge WWTP"
raw = PiDataRetrival(site_name).copy()
raw

OSIsoft(r) AF SDK Version: 2.10.8.440


C:\Users\ka005254\AppData\Roaming\Python\Python311\site-packages\PIconnect\PIAF.py:77: InitialisationWarning: Failed loading server data for ods with error EndpointNotFoundException
  servers: dict[str, ServerSpec] = _lookup_servers()


,Site,Attribute,Timestamp,Value
0,Murray Bridge WWTP,1 - Influent,2020-09-01,80.866989
1,Murray Bridge WWTP,1 - Influent,2020-10-01,86.926613
2,Murray Bridge WWTP,1 - Influent,2020-11-01,79.002815
3,Murray Bridge WWTP,1 - Influent,2020-12-01,79.807335
4,Murray Bridge WWTP,1 - Influent,2021-01-01,80.080437
...,...,...,...,...
172,Murray Bridge WWTP,3 - Reuse,2025-03-01,87.581718
173,Murray Bridge WWTP,3 - Reuse,2025-04-01,82.271561
174,Murray Bridge WWTP,3 - Reuse,2025-05-01,85.485222
175,Murray Bridge WWTP,3 - Reuse,2025-06-01,77.001717
